# LIN340 Project Notebook

## Colab Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
%cd /content/drive/MyDrive/LIN340Project

In [ ]:
!pip install transformers peft datasets bert-score rouge-score nltk accelerate sentencepiece stanza

## Preprocessing

In [ ]:
import re, os, random, stanza

# Italian stanza models downloaded for tokenizer, parts of speech, lemmatization, and constituency phrase structure trees.
stanza.download('it', processors='tokenize,pos,lemma,constituency', verbose=False)
nlp = stanza.Pipeline('it', processors='tokenize,pos,lemma,constituency', verbose=False)

def remove_punctuation(text):
    # Remove any character that is not a word character (letter/digit/underscore) or whitespace.
    return re.sub(r'[^\w\s]', '', text)

def read_novel(path):
    # Read raw novel text and use latin-1 for accented characters.
    with open(path, encoding='latin-1') as f:
        txt = f.read()
    # Newlines and extra whitespace converted to single space.
    txt = re.sub(r'\n', ' ', txt)
    txt = re.sub(r'\s+', ' ', txt)
    # Lowercase
    return txt.lower()

# Map stanza constituency labels to bracket token names, keeping them short. 
TAGS = {'NP': 'NP', 'VP': 'VP', 'PP': 'PP', 'ADJP': 'AP', 'ADVP': 'ADVP'}

def render_tree(node, word_iter):
    # Recursively go through constituency parse tree to produce tagged string, word_iter is a shared iterator that is used as leaves are encountered.
    if node.is_leaf():
        # Leaf nodes are actual words, pull next one from iterator. 
        word = next(word_iter, None)
        if word is None:
            return ''
        return remove_punctuation(word.text)
    # Render all children first in internal node. 
    parts = []
    for child in node.children:
        p = render_tree(child, word_iter)
        if p.strip():
            parts.append(p)
    text = ' '.join(parts)
    if not text.strip():
        return ''
    # Wrap text in bracket tokens if node is a target phrase type.
    if node.label in TAGS:
        tag = TAGS[node.label]
        return f'[{tag}_START] {text} [{tag}_END]'
    # Other node types (S, ROOT, etc.) passed without tagging. 
    return text

def process(text, chunk_size=50000):
    # Process novel in chunks, prevents running out of memory with Stanza.
    # Return parallel lists, a raw and a tagged sentence for each entry.
    raw_sents = []
    tagged_sents = []
    i = 0
    while i < len(text):
        chunk = text[i:i + chunk_size]
        # Trim to last space so mid-word is not cut when getting chunks.
        if i + chunk_size < len(text):
            last_space = chunk.rfind(' ')
            if last_space > 0:
                chunk = chunk[:last_space]
        doc = nlp(chunk)
        for sent in doc.sentences:
            # Build raw version, tokens without punctuation. 
            tokens = [t for t in [remove_punctuation(w.text) for w in sent.words] if t.strip()]
            raw = ' '.join(tokens).strip()
            # Skip very small sentences as they might be artifacts.
            if len(raw) <= 10:
                continue
            raw_sents.append(raw)
            # Build tagged version by going through constituency tree.
            tagged = re.sub(r'\s+', ' ', render_tree(sent.constituency, iter(sent.words))).strip()
            # If trees outputs nothing, use raw sentence instead.
            tagged_sents.append(tagged if tagged else raw)
        i += len(chunk)
    return raw_sents, tagged_sents

txt = read_novel('q.txt')
txt2 = read_novel('guerra_agli_umani.txt')
print(f'q: {len(txt)} chars, guerra: {len(txt2)} chars')

q_raw, q_tagged = process(txt)
print(f'q: {len(q_raw)} sentences')

g_raw, g_tagged = process(txt2)
print(f'guerra: {len(g_raw)} sentences')

# Combine both novels, zip them to keep them paired, and shuffle. 
# Fixed seed for reproducibility.
combined = list(zip(q_raw + g_raw, q_tagged + g_tagged))
random.seed(42)
random.shuffle(combined)
all_raw = [x[0] for x in combined]
all_tagged = [x[1] for x in combined]
print(f'total: {len(all_raw)} pairs')


In [ ]:
n = len(all_raw)
# Train validation test split boundaries
i80, i90 = int(n * .8), int(n * .9)

train_raw, val_raw, test_raw = all_raw[:i80], all_raw[i80:i90], all_raw[i90:]
train_tagged, val_tagged, test_tagged = all_tagged[:i80], all_tagged[i80:i90], all_tagged[i90:]
print(f'Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}')

os.makedirs('data', exist_ok=True)

# Save each split, sentence per line.
for split, raw, tagged in [('train', train_raw, train_tagged),
                             ('val', val_raw, val_tagged),
                             ('test', test_raw, test_tagged)]:
    with open(f'data/raw_{split}.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(raw))
    with open(f'data/tagged_{split}.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(tagged))
print('wrote data files')


# Build eval prompt and reference pairs from test set. Keep it long enough >= 15 words, and split 2/3 into the prompt and keep 1/3 as reference.
# Use 20 pairs in total.
pairs = []
for sent in test_raw:
    words = sent.split()
    if len(words) < 15:
        continue
    cut = int(len(words) * 2 / 3)
    pairs.append(' '.join(words[:cut]))   # prompt
    pairs.append(' '.join(words[cut:]))   # reference
    if len(pairs) >= 40:
        break

with open('data/eval_prompts.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(pairs))
print(f'wrote eval_prompts.txt ({len(pairs)//2} pairs)')


## Training

In [ ]:
!python train.py raw

In [ ]:
!python train.py tagged

## Evaluation

In [ ]:
!python evaluate.py

## Results

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('results/eval_results.json') as f:
    data = json.load(f)

# Indexing by mode for easier access to results
res = {r['mode']: r for r in data}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Raw vs Tagged GPT-2 — Evaluation Metrics')

# BLEU scores
x = np.arange(4)  # four variants
ax = axes[0]
# Offset the two bars by 0.2, otherwise too much overlap.
ax.bar(x - 0.2, [res['raw'][k] for k in ['bleu_1','bleu_2','bleu_3','bleu_4']], 0.4, label='RAW')
ax.bar(x + 0.2, [res['tagged'][k] for k in ['bleu_1','bleu_2','bleu_3','bleu_4']], 0.4, label='TAGGED')
ax.set_title('BLEU')
ax.set_xticks(x)
ax.set_xticklabels(['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4'])
ax.legend()

# ROUGE and BERTScore F1
ax = axes[1]
ax.bar(x - 0.2, [res['raw'][k] for k in ['rouge1','rouge2','rougeL','bertscore_f1']], 0.4, label='RAW')
ax.bar(x + 0.2, [res['tagged'][k] for k in ['rouge1','rouge2','rougeL','bertscore_f1']], 0.4, label='TAGGED')
ax.set_title('ROUGE & BERTScore F1')
ax.set_xticks(x)
ax.set_xticklabels(['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore F1'])
ax.legend()

# Perplexity.
ax = axes[2]
ax.bar(['RAW', 'TAGGED'], [res['raw']['perplexity'], res['tagged']['perplexity']])
ax.set_title('Perplexity (lower = better)')

plt.tight_layout()
plt.savefig('results/metrics_comparison.png')
plt.show()


In [ ]:
import json

with open('results/eval_results.json') as f:
    data = json.load(f)

# Unpack results dictinoary for raw and tagged.
raw, tagged = data

# Side by side metric comparison.
metrics = ['perplexity', 'bleu_1', 'bleu_2', 'bleu_3', 'bleu_4',
           'rouge1', 'rouge2', 'rougeL', 'bertscore_p', 'bertscore_r', 'bertscore_f1']
for m in metrics:
    print(f'{m}: raw={raw[m]:.4f}, tagged={tagged[m]:.4f}')


## References

The following repositories were used as reference points for the design and implementation of this project.

**[wietsedv/gpt2-recycle](https://github.com/wietsedv/gpt2-recycle/tree/master/src)**
Presents a multi-stage vocabulary recycling method that transfers English GPT2 weights to Italian and Dutch without full retraining, producing the `GroNLP/gpt2-small-italian` model used as the base model here. Referenced for the base model and Italian GPT2 adaptation. 


**[google-deepmind/transformer_grammars](https://github.com/google-deepmind/transformer_grammars)**
Implements a language modeling architecture that incorporates syntactic constituency structure, by treating linearized parse tree operations as special vocab tokens with structure dependent attention masks. Referenced as the primary prior work for injecting constituency brackjet tokens (NP, VP, PP, etc.) into language model training data, which the tagged corpus we use is a lighter version of.

**[Yinghao-Li/GuiGen](https://github.com/Yinghao-Li/GuiGen)**
A nenural text generation system that first expands a syntactic parse structure, then generates text conditioned on it, using constituency tree guidance to improve generation quality. A supporting motivation for using constituency structure explicitly during language model training.

**[kennethugo/PEFT-LoRA_Fine-tuning](https://github.com/kennethugo/PEFT-LoRA_Fine-tuning)**
Uses HuggingFace PEFT with LoRA to fine-tune Flan-T5 for dialogue summarization, comparing base model against LoRA-adapted variant using ROUGE metrics to show that parameter efficieint fine-tuning achieves comparable performance at lower computational cost. Referenced both for the LoRA adaptation and for comparing two model varaints with overlapping evaluation metrics, such as the raw vs. tagged comparison done here.
